In [185]:
import sys
import os
# Add the local stonesoup directory to sys.path
project_path = r"C:\Users\joesb\Documents\stonesoup"  # Adjust this to your actual path
if project_path not in sys.path:
    sys.path.insert(0, project_path)
import pandas as pd

In [231]:
num_timesteps= 10000
number_particles = 2000
measurement_sigma2=1

sigma_W2 = 1e-2**2
q=1e-4
alpha = 1.1

In [232]:

from datetime import datetime, timedelta
from sre_parse import State
from stonesoup.types.detection import Detection
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel

import numpy as np
from datetime import timedelta

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.particle import Particle
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState, ParticleState
from stonesoup.types.array import StateVectors

# Step 1: Load the CSV file
excel_file = fr"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSD_exchange_rate_tracker\EURtoUSD_Historical_ExchaNge_Rates.xlsx"  # Replace with your file path
data = pd.read_excel(excel_file)

# Step 2: Extract relevant columns
discrete_time = data['Discrete_Time']
prices = data['Price']
dp_dt = data['dP/dt']

num_timesteps= min(num_timesteps,len(discrete_time))
discrete_time=discrete_time[:num_timesteps]
prices=prices[:num_timesteps]
dp_dt=dp_dt[:num_timesteps]

# Step 3: Generate Timestamps
start_time = datetime.now().replace(microsecond=0)
timesteps = [start_time + timedelta(seconds=int(dt)) for dt in discrete_time]

# Step 4: Build the Observed Prices as noisy measurements
from stonesoup.models.measurement.linear import LinearGaussian

# Define Measurement Model
measurement_model = LinearGaussian(
    ndim_state=2,  # State vector dimensions: [price, dP/dt]
    mapping=(0,),  # Map the measurement to the 'price' dimension
    noise_covar=np.diag([measurement_sigma2])  # Small measurement noise for 'price'
)

# Generate Measurements from Ground Truth
plottable_observations=Track()
measurements = []
for i in range(len(timesteps)):
    state_vector = [prices[i], dp_dt[i]]  # Price as the first axis, dp/dt as the second
    timestamp = timesteps[i]
    state=GroundTruthState(state_vector=state_vector,timestamp=timestamp)
    plottable_observations.append(state)
    measurement = measurement_model.function(state, noise=False) #we want our observed price to be what we view as the noisy observation
    measurements.append(Detection(
        measurement,
        timestamp=timestamp,
        measurement_model=measurement_model
    ))

In [233]:
#MPF track
seed = 1 # Random seem for reproducibility
# Driving process parameters
mu_W = (prices[num_timesteps-1]-prices[0])/num_timesteps
c=10

noise_case=NoiseCase(2)

# Model parameters
theta=0.15

driver_x = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, noise_case=noise_case)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta, mu_W=mu_W)
transition_model = CombinedLinearLevyTransitionModel([langevin_x])

predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)


# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(
    mean=np.array([prices[0],dp_dt[0]]),  # Initial state: [price, dP/dt]
    cov=np.diag([1.0, 1.0]),  # Covariance for the initial state
    size=number_particles
)

# Define covariance for particles
covars = np.stack(
    [np.eye(2) * 100 for _ in range(number_particles)], axis=2
)  # Shape: (2, 2, number_particles)

# Create prior particle state
prior = MarginalisedParticleState(
    state_vector=StateVectors(states.T),  # Transpose states to shape (2, N)
    covariance=covars,  # Covariance matrix
    weight=np.array([Probability(1 / number_particles)] * number_particles),
    timestamp=start_time-timedelta(seconds=1)
)

from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

LP_track = Track()

for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis)
    LP_track.append(post)
    prior = LP_track[-1]
    print(f"track length ={len(LP_track)} of {len(measurements)}")

# from stonesoup.smoother.particle import MarginalisedKalmanSmoother, ParticleSmoother, CarterKohnSampler
# particlesmoother=ParticleSmoother()
# culled_track=particlesmoother.particle_paths(track=track)

# RTSsmoother=MarginalisedKalmanSmoother()
# RTS_track=RTSsmoother.smooth(culled_track=culled_track)

# CKsmoother=CarterKohnSampler(MCMCsample=None,measurements=measurements)
# CK_track=CKsmoother.smooth(culled_track=culled_track)

track length =1 of 2610
track length =2 of 2610
track length =3 of 2610
track length =4 of 2610
track length =5 of 2610
track length =6 of 2610
track length =7 of 2610
track length =8 of 2610
track length =9 of 2610
track length =10 of 2610
track length =11 of 2610
track length =12 of 2610
track length =13 of 2610
track length =14 of 2610
track length =15 of 2610
track length =16 of 2610
track length =17 of 2610
track length =18 of 2610
track length =19 of 2610
track length =20 of 2610
track length =21 of 2610
track length =22 of 2610
track length =23 of 2610
track length =24 of 2610
track length =25 of 2610
track length =26 of 2610
track length =27 of 2610
track length =28 of 2610
track length =29 of 2610
track length =30 of 2610
track length =31 of 2610
track length =32 of 2610
track length =33 of 2610
track length =34 of 2610
track length =35 of 2610
track length =36 of 2610
track length =37 of 2610
track length =38 of 2610
track length =39 of 2610
track length =40 of 2610
track len

In [234]:
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater


seed=1
q=1e-4
transition_model=ConstantVelocity(noise_diff_coeff=q)
predictor=KalmanPredictor(transition_model)
updater=KalmanUpdater(measurement_model=measurement_model)
prior=GaussianState(state_vector==np.array([prices[0],0]),
                    covar=np.diag([1000.0,1000.0]),
                    timestamp=start_time-timedelta(seconds=1))

CV_track = Track()

for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis)
    CV_track.append(post)
    prior = CV_track[-1]


In [235]:
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSDplots"

In [236]:
from matplotlib import axis


axis_label_list=["Price"]
particle_plotter_dict = {}
uncertainty=False
particle=False
plot_particle_paths=False

#TODO: improve the uncertainty legend and add an 'individual_tracks' option for particle plotting which plots track[t=0:T-1][i] for all i
i=0
label=axis_label_list[i]

file_path = Path(folder_path + rf"\1D_plot_{label}_{num_timesteps}steps_{number_particles}p.html")
file_path.parent.mkdir(parents=True, exist_ok=True)

particle_plotter_dict[label]= Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.ONE, axis_labels=[label])
if label =="Price":
    particle_plotter_dict[label].plot_ground_truths(plottable_observations, [i], truths_label="Price Observations")


particle_plotter_dict[label].plot_tracks(LP_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Levy Process",line=dict(width=1))
particle_plotter_dict[label].plot_tracks(CV_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Constant Velocity",line=dict(width=1))

particle_plotter_dict[label].fig.update_layout( 
    plot_bgcolor="white",  # Set background color to white
    xaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text="Time", font=dict(size=20)),  # Add large label
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text=label, font=dict(size=20)),  # Add large label
    ),
    legend=dict(
        font=dict(size=15),       # Make the legend font larger
        # orientation='v',
        # xanchor="auto",         # Center the legend
        # yanchor="auto",           # Align the legend to the bottom of the plot
        bordercolor="Black",
        borderwidth=3,
        # y=+0.45,                   # Position it above the graph
        # x=0.6                    # Center it horizontally
    ),
)
particle_plotter_dict[label].fig.write_html(str(file_path))
particle_plotter_dict[label].fig.show()